## Visualización de los rCMBs a partir del excel
Correcciones:
- Se leen en formato (x,y,z)
- Están en formato 1-based, se cambian a 0-based para que sea compatible con python

In [13]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# RUTAS A AJUSTAR
# =========================
EXCEL_PATH = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMBInformationInfo.xlsx"
NIFTI_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/rCMB_DefiniteSubject/data"
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/CSIRO/CSIRO_dataset/visualize_rCMBs_from_excel_corrected"

os.makedirs(OUTPUT_DIR, exist_ok=True)


def safe_int(val):
    """Convierte a int manejando cualquier basura"""
    if pd.isna(val) or str(val).strip() == '':
        return None
    try:
        return int(float(str(val).strip()))
    except:
        return None

# ========================= PROCESAR =========================
df = pd.read_excel(EXCEL_PATH)

total_cmbs = 0
valid_cmbs = 0

for idx, row in df.iterrows():
    nifti_name = str(row.iloc[0]).strip()  # COLUMNA 0: nombre archivo
    if not nifti_name.endswith('.nii.gz'):
        continue

    nifti_path = os.path.join(NIFTI_DIR, nifti_name)
    if not os.path.exists(nifti_path):
        print(f"{nifti_name}")
        continue

    print(f"\n{nifti_name}")
    img = nib.load(nifti_path)
    data = img.get_fdata()
    shape = data.shape
    print(f"   Shape: {shape}")

    # *** EXTRACCIÓN SIMPLE: grupos de 3 desde columna 1 ***
    coords = []

    for i in range(1, len(row), 3):  # Empezar en columna 1, saltar de 3 en 3
        if i+2 < len(row):
            x_raw = safe_int(row.iloc[i])     # Col i:   x
            y_raw = safe_int(row.iloc[i+1])   # Col i+1: y  
            z_raw = safe_int(row.iloc[i+2])   # Col i+2: z
            
            # *** PRIMERO verificar que NO es None, DESPUÉS restar 1 ***
            if x_raw is not None and y_raw is not None and z_raw is not None:
                x = x_raw - 1
                y = y_raw - 1  
                z = z_raw - 1
                
                # Verificar que no sea negativo
                if x >= 0 and y >= 0 and z >= 0:
                    coords.append((x, y, z))

    print(f"   CMBs encontrados: {len(coords)}")
    total_cmbs += len(coords)

    # *** DIBUJAR CADA CMB COMO CIRUNFERENCIA ROJA***
    for cmb_idx, (x, y, z) in enumerate(coords, 1):
        if not (0 <= x < shape[0] and 0 <= y < shape[1] and 0 <= z < shape[2]):
            print(f"   CMB{cmb_idx} fuera: ({x},{y},{z})")
            continue

        valid_cmbs += 1
        slice_img = data[:, :, z]

        fig, ax = plt.subplots(figsize=(8, 8))
        
        # Imagen con buen contraste
        vmin = slice_img.mean() - slice_img.std()
        vmax = slice_img.mean() + 2*slice_img.std()
        ax.imshow(slice_img.T, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
        
        # *** SOLO CIRCUNFERENCIA ROJA VACÍA (sin punto central) ***
        radius = 5
        circle = plt.Circle((x, y), radius, color='red', fill=False, linewidth=2)
        ax.add_patch(circle)
        
        ax.set_title(f"{os.path.splitext(nifti_name)[0]}\nCMB #{cmb_idx} (x={x},y={y},z={z})", 
                    fontsize=14, pad=20)
        ax.axis('off')

        out_name = f"{os.path.splitext(nifti_name)[0]}_CMB{cmb_idx}_z{z:03d}.png"
        plt.savefig(os.path.join(OUTPUT_DIR, out_name), bbox_inches='tight', dpi=200, facecolor='white')
        plt.close(fig)
        
        print(f"   CMB{cmb_idx}: ({x},{y},{z}) → {out_name}")

print(f"\nTOTAL CMBs válidos: {valid_cmbs} (buscados: 148)")


122_T1_MRI_SWI_BFC_50mm_HM.nii.gz
   Shape: (176, 256, 80)
   CMBs encontrados: 2
   CMB1: (63,174,11) → 122_T1_MRI_SWI_BFC_50mm_HM.nii_CMB1_z011.png
   CMB2: (124,133,43) → 122_T1_MRI_SWI_BFC_50mm_HM.nii_CMB2_z043.png

122_T2_MRI_SWI_BFC_50mm_HM.nii.gz
   Shape: (176, 256, 80)
   CMBs encontrados: 5
   CMB1: (50,166,25) → 122_T2_MRI_SWI_BFC_50mm_HM.nii_CMB1_z025.png
   CMB2: (64,170,12) → 122_T2_MRI_SWI_BFC_50mm_HM.nii_CMB2_z012.png
   CMB3: (131,101,38) → 122_T2_MRI_SWI_BFC_50mm_HM.nii_CMB3_z038.png
   CMB4: (130,122,42) → 122_T2_MRI_SWI_BFC_50mm_HM.nii_CMB4_z042.png
   CMB5: (125,128,44) → 122_T2_MRI_SWI_BFC_50mm_HM.nii_CMB5_z044.png

218_T0_MRI_SWI_BFC_50mm_HM.nii.gz
   Shape: (176, 256, 80)
   CMBs encontrados: 2
   CMB1: (73,174,12) → 218_T0_MRI_SWI_BFC_50mm_HM.nii_CMB1_z012.png
   CMB2: (142,149,55) → 218_T0_MRI_SWI_BFC_50mm_HM.nii_CMB2_z055.png

218_T2_MRI_SWI_BFC_50mm_HM.nii.gz
   Shape: (176, 256, 80)
   CMBs encontrados: 1
   CMB1: (74,176,13) → 218_T2_MRI_SWI_BFC_50mm_HM.n